In [1]:
import pandas as pd

df = pd.read_excel("/content/RealAndSynthetic.xlsx")

df.head()

,review_text,issue
0,I purchased these sheets based on positive Wir...,Material_Quality
1,Picked up a full sheet set from the store. Pac...,Size_Fit
2,Did not like how they came out of the washer a...,Comfort_Softness
3,"I’ve washed these twice, using detergent and l...",Chemical_Odor
4,Terrible chemical smell. Possibly formaldehyde...,Chemical_Odor


In [2]:
df.sample(10)

,review_text,issue
4806,The product was shipped with no padding inside...,Shipping_Damage
2948,Sheets are of poor quality and not the standar...,Other
4504,They deducted a 're-stocking fee' of 15% even ...,Return_Refund
409,these sheets are very scratchy and uncomfortable,Comfort_Softness
2330,Surprised by how thin these are,Material_Quality
1034,"Really disappointed with these sheets. First, ...",Defect
1927,The Threshold Performance sheets that I bought...,Material_Quality
1738,I bought these sheets based on a recommendatio...,Comfort_Softness
2709,Unfortunately when I got home ( an hour drive ...,Wrong_Item
3918,The extra buttons for the shirt were missing.,Missing_Parts


In [3]:
df.isnull().sum()

,0
review_text,0
issue,1


In [4]:
df.shape

(5549, 2)

In [5]:
df = df.dropna()

In [6]:
df.shape

(5548, 2)

In [7]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["label"] = le.fit_transform(df["issue"])

In [8]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [9]:
train_df.shape, test_df.shape

((4438, 3), (1110, 3))

In [10]:
train_df

,review_text,issue,label
4592,There was absolutely zero padding inside the s...,Packaging,11
446,"Unfortunately, I don't know how the product is...",Missing_Parts,9
88,These are nice and I chose for the natural fib...,Chemical_Odor,0
4670,The outer carton looked like it had been dropp...,Packaging,11
4212,I was charged a 're-stocking and handling' fee...,Return_Refund,14
...,...,...,...
2378,"Liked the material. However, the ottom sheet i...",Size_Fit,17
3142,These sheets pill so badly that after washing ...,Pilling,12
1454,This is my second set of these sheets. I have ...,Size_Fit,17
1662,I didn’t look at other reviews before buying a...,Wrinkle_Resistance,18


In [11]:
pip install transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


In [12]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [13]:
def tokenize(batch):
    return tokenizer(
        batch["review_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [14]:
# coverting dataset HF Dataset
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

In [15]:
train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

Map:   0%|          | 0/4438 [00:00<?, ? examples/s]

Map:   0%|          | 0/1110 [00:00<?, ? examples/s]

In [16]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=20
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01
)

In [18]:
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted"
        )
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [19]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,1.349470,0.645946,0.644672,0.645946,0.607341
2,1.642158,0.822264,0.780180,0.788409,0.780180,0.768280
3,1.642158,0.666655,0.797297,0.807337,0.797297,0.796434
4,0.532704,0.633960,0.803604,0.802731,0.803604,0.798647
5,0.532704,0.649589,0.824324,0.827518,0.824324,0.824295
6,0.199469,0.742862,0.815315,0.814959,0.815315,0.811494
7,0.199469,0.757450,0.821622,0.822386,0.821622,0.819207
8,0.075542,0.802930,0.824324,0.820208,0.824324,0.819499
9,0.022597,0.844296,0.821622,0.821366,0.821622,0.819038
10,0.022597,0.871293,0.825225,0.820757,0.825225,0.820299


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4170, training_loss=0.2982416290459301, metrics={'train_runtime': 955.592, 'train_samples_per_second': 69.664, 'train_steps_per_second': 4.364, 'total_flos': 2205296364902400.0, 'train_loss': 0.2982416290459301, 'epoch': 15.0})

In [21]:
from sklearn.metrics import (
    classification_report
)

predictions = trainer.predict(
    test_ds
)

y_pred = np.argmax(
    predictions.predictions,
    axis=1
)

y_true = predictions.label_ids

print(
    classification_report(
        y_true,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.78      0.85      0.81        41
           1       0.92      0.87      0.89        38
           2       0.90      0.92      0.91       212
           3       0.95      0.90      0.92        40
           4       0.65      0.72      0.68        39
           5       0.95      0.98      0.96        41
           6       0.62      0.56      0.59        32
           7       0.82      0.84      0.83        80
           8       0.58      0.50      0.54       110
           9       0.89      0.86      0.88        29
          10       0.68      0.65      0.67        26
          11       0.93      0.93      0.93        40
          12       0.75      0.81      0.78        57
          13       0.97      0.70      0.81        40
          14       0.90      0.95      0.93        78
          15       0.89      0.98      0.93        43
          16       0.92      0.90      0.91        40
          17       0.79    

In [22]:
trainer.save_model(
    "issue_classifier"
)

tokenizer.save_pretrained(
    "issue_classifier"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('issue_classifier/tokenizer_config.json', 'issue_classifier/tokenizer.json')

In [23]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

tokenizer = AutoTokenizer.from_pretrained(
    "issue_classifier"
)

model = AutoModelForSequenceClassification.from_pretrained(
    "issue_classifier"
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [24]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer
)

review = """
The package arrived crushed and the box was torn.
"""

result = classifier(review)

print(result)

[{'label': 'LABEL_15', 'score': 0.9979755282402039}]


In [25]:
print(le.classes_)

['Chemical_Odor' 'Color_Appearance' 'Comfort_Softness' 'Customer_Service'
 'Defect' 'Delivery_Delay' 'Design_Issue' 'Durability' 'Material_Quality'
 'Missing_Parts' 'Other' 'Packaging' 'Pilling' 'Positive' 'Return_Refund'
 'Shipping_Damage' 'Shrinkage' 'Size_Fit' 'Wrinkle_Resistance'
 'Wrong_Item']


In [26]:
import torch

def predict_issue_with_confidence(text):

    device = model.device

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model(**inputs)

        probs = torch.softmax(
            outputs.logits,
            dim=1
        )

    pred_idx = probs.argmax().item()

    confidence = probs.max().item()

    issue = le.inverse_transform(
        [pred_idx]
    )[0]

    return issue, confidence

In [29]:
predict_issue_with_confidence(
    "There is hole in the sheet"
)

('Defect', 0.6603882312774658)

In [28]:
print(model.device)

cuda:0
